# Reto Coppel

por: 
- Mateo Yáñez Tanaka A01639199
- Fernando Ramos Valdez A00228153
- Fernando Pavía González A01639317
- Pedro Manuel Montes Valle A01741659
- Jose Luis Santos Montaño A01781721

## Introducción

En este reto se nos pidió que, usando datos proveeídos por coppel, modelaramos la forma óoptima de enviar dinero a diferentes locaciones para poder así minimizar los costos de esta operación. Los datos que nos fueron entregados están separados en tres archivos ```tiendasGDL.csv```, ```ventas.csv``` y ```abonos.csv```. Aúnado a esto nos fueron entregadas los siguientes supuestos y restricciones a considerar para completar el reto:
* **Supuestos**: 
    * Se omite el rol del coorresponsal bancario y la cobranza domiciliaria
    * Se ignora que la central proporcionará las claves de ropa y tienda
    * Se intenta modelar el flujo de dinero entre tiendas, tanto efectivo como electrónico
* **Restricciones**:
    * Ninguna tienda debe de tener más de $250,000 en efectivo
    * Se requieren al menos 100 monedas y billetes de cada denominación (50, 20, 10, 5, 2, 1). Las necesidades de efectivo son mayores en ropa, seguida por calzado y finalmente muebles.
    * A la central se le deben de depositar 30% de las ventas de manera electrónica.

Finalmente se nos pidió que tomasemos en cuenta los siguientes costos operacionales:
* Costo de camioneta de valores: 25 pesos por kilómetro recorrido y 3 pesos de seguro
por cada por cada $1,000 pesos de efectivo trasladado, pudiendo transportar hasta
100,000 por viaje.
* Costo de recolección de efectivo: 0.5% de las ventas en efectivo.
* Costo de abonos: 0.5% de los abonos.
* Comisiones por transferencias electrónicas: 0.1%
* Las ventas a crédito implica dar un abono inicial en efectivo del 15%

## Planteamiento del problema:
Para poder comenzar a plantear el prolema debemos de ver lo que tenemos.



In [15]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import itertools as it
import matplotlib.pyplot as plt
from math import cos, radians
import matplotlib.cm as cm

tiendas = pd.read_csv('tiendasGDL.csv', index_col= 'TiendaCodigo')
ventas = pd.read_csv('ventas.csv', index_col= 'num_tie')
abonos = pd.read_csv('abonos.csv', index_col = 'tienda')

In [16]:
no_sale = []
for index in tiendas.index:
    count = 0
    for i in ventas.index:
        if index == i:
            count += 1
    if count == 0:
        no_sale.append(index)
    print(f'Registros de ventas para la tienda {index}: {count}')

Registros de ventas para la tienda 501: 0
Registros de ventas para la tienda 503: 47
Registros de ventas para la tienda 511: 49
Registros de ventas para la tienda 512: 49
Registros de ventas para la tienda 82: 49
Registros de ventas para la tienda 84: 49
Registros de ventas para la tienda 86: 49
Registros de ventas para la tienda 550: 0
Registros de ventas para la tienda 1050: 48
Registros de ventas para la tienda 1091: 0
Registros de ventas para la tienda 6548: 49
Registros de ventas para la tienda 299: 49
Registros de ventas para la tienda 758: 49
Registros de ventas para la tienda 1234: 0
Registros de ventas para la tienda 126: 0
Registros de ventas para la tienda 144: 49
Registros de ventas para la tienda 615: 0
Registros de ventas para la tienda 619: 0
Registros de ventas para la tienda 1113: 0
Registros de ventas para la tienda 1159: 49
Registros de ventas para la tienda 419: 49
Registros de ventas para la tienda 1374: 0
Registros de ventas para la tienda 1389: 49
Registros de ve

In [17]:
no_transfer = []
for index in tiendas.index:
    count = 0
    for i in abonos.index:
        if index == i:
            count += 1
    if count == 0:
        no_transfer.append(index)
    print(f'Registros de abonos para la tienda {index}: {count}')

Registros de abonos para la tienda 501: 0
Registros de abonos para la tienda 503: 28
Registros de abonos para la tienda 511: 28
Registros de abonos para la tienda 512: 28
Registros de abonos para la tienda 82: 28
Registros de abonos para la tienda 84: 28
Registros de abonos para la tienda 86: 28
Registros de abonos para la tienda 550: 0
Registros de abonos para la tienda 1050: 28
Registros de abonos para la tienda 1091: 0
Registros de abonos para la tienda 6548: 28
Registros de abonos para la tienda 299: 28
Registros de abonos para la tienda 758: 28
Registros de abonos para la tienda 1234: 0
Registros de abonos para la tienda 126: 0
Registros de abonos para la tienda 144: 28
Registros de abonos para la tienda 615: 0
Registros de abonos para la tienda 619: 0
Registros de abonos para la tienda 1113: 0
Registros de abonos para la tienda 1159: 28
Registros de abonos para la tienda 419: 28
Registros de abonos para la tienda 1374: 0
Registros de abonos para la tienda 1389: 28
Registros de ab

In [18]:
no_sale

[501, 550, 1091, 1234, 126, 615, 619, 1113, 1374, 1216, 1217, 1324, 1475]

Lo primero que es necesario notar es que hay 13 tiendas que no tienen registros, estas son la tiendas ubicada en 'Los Ángeles', 'Teran', 'Alcalde', '16 de Septiembre', 'Corona', 'Estadio', 'Mercado Bola', 'Oblatos', 'Echeverría', 'Bodega Liquidación', 'Gobernador', 'Plaza Revolución' y 'Plaza Las Torres', tomando esto en cuenta se decidirá ignorar estas tiendas en la formulación del problema. Algo que también es relevante notar es que casi todas las tiendas tienen 49 registros de ventas y 28 registros de abonos

In [19]:
tiendas.drop(no_sale, axis='index', inplace=True)

In [20]:
tiendas['TipoTienda'].value_counts()

TipoTienda
Coppel         11
Canada          8
Minitiendas     2
Name: count, dtype: int64

Lo siguiente que es relevante notar es que las tiendas de tipo coppel están desproporcionalmente representadas con respecto a las otras dos clases que son canadá y minitienda con 11 siendo de tipo Coppel, 8 de tipo Canada  y 2 Minitiendas.

Con esto ya comprendido se puede comenzar propiamente con la formulación. La formulación estará basada en un VRP, sin emargo tendrá unas cuantas diferencias clave, las tiendas no necesariamente deben ser visitadas, pero si lo son será cuando mucho una vez al día, hay un costo por el flujo de efectivo, y otro costo por las transferencias realizadas (Se entrará más a detalle en estos aspectos del modelo más adelante). Para plantear el modelo tomamos en cuenta los siguientes **supuestos**:
- Para simplificar el problema, pero mantener un estimado realista, se asumirá que las distancias entre tiendas son la distancia manhattan de sus cooordenadas convertidas a kilómetros
- Se asumirá que la cantidad de camiones es infinita puesto que no se sabe con exactitud cuántos camiones poseé Coppel para este tipo de tarea
- Para simplificar el problema se asume que la distribución de  las denominaciones transportadas por el camión es uniforme. 
- Se asume que los camiones debido a que son de necesidad especifica provienen de un servicio contratado como [SEGURITEC](https://www.gruposeguritec.com.mx/) y por ende los movimientos pueden ser de tienda a tienda
- Se asume que la central es el CEDI que se encuentra en Dr.Roberto Michel 970
- Se asume que el cedi no puede enviar dinero debidoo a la incertidumbre del costo de recolección
- Se asume que los 8800 pesos que requiere cada tienda se distribuyen uniformemente en las denominaciones que necesitan (100 monedas/billetes de 50, 20, 10, 5, 2,1)
- Para simplificar el problema se asumirá que estos viajes se hacen únicamente una vez por día y se ignorarán las ventanas de tiempo durante el día (el problema se resolverá para cada día de la semana)
- Se asume que el 30% de las ventas que se transfiere a la central se transfiere diariamente
- Para simplificar el modelo se asume que el 30% de las ventas que se transfiere es únicamente de las ventas en efectivo
- Se asumirá que el dinero en tienda al comenzar la solución será únicamente las ventas del día previo al inicio de la solución, después el valor del dinero en tienda será estimado.
-Para simplificar el modelo se asumirá que todas las ventas son unicamente aquellas de efectivo
- Debido a que el problema se enfoca en el flujo de efectivo se ignorarán las transferencias exceptuando el 30% que se transfiere diariamente a la central y transferencias realizadas a la central. Esto es debido a que a pesar de que las transferencias no son una buena alternativa para mover efectivo entre sucursales, sí funcionan para deshacerse de él y enviarlo a la central que no requiere efectivo. 

Tomando esto y las restricciones dadas por coppel en cuenta llegamos a la siguiente formulación para el problema 
$$Z=25 \sum_{i=1}^n \sum_{j=1}^nC_{ij}X_{ij} + \frac{3}{1000}\sum_{j=1}^n\sum_{i=1}^nF_{ij} + \frac{5}{1000}\sum_{i=2}^n\sum_{j=1}^nX_{ij}V_{i} + 0.01T_{i}$$
Los **parámetros** del modelo serían los siguientes:
- $n$  = Número de puntos ( $n$-1 tiendas + 1 central)
- $C_{ij}$ = La distancia entre la tienda i a la tienda j
- $P_{i}$ = Dinero en la tienda i
- $V_{i}$ = Ventas en la tienda i
- $D_{i}$ = Demanda de la tienda i (8,800 pesos) 
- $C$  = Capacidad del camión (100,000 pesos)

Las **variables de decisión** serían las siguientes:
- $X_{ij}$ (Binaria)
    - 1 : si un camión va de la tienda $i$ a la tienda $j$
    - 0 : en caso contrario
- $F_{ij}$ = Flujo de dinero transportada de la tienda $i$ a la tienda $j$
- $T_{i}$ = Transferencias de la tienda $i$ a la central

Las **rstricciones** de este problema son las siguientes:
- Se requieren al menos 100 monedas o billetes de cada denominación (50, 20, 10, 5, 2 y 1 peso) $$\sum_{j=1}^nF_{ji}-\sum_{j=1}^nF_{ij} +P{i} \ge 8 800;  i=2, ... , n$$

- Ninguna tienda debe de tener más de 250,000 pesos $$\sum_{j=1}^nF_{ji}-\sum_{j=1}^nF_{ij} + P_{i} \le 250 000; i=2, ... , n$$

- Los vehículos no pueden cargar más de 100,000 pesos
$$0 \le Fij \le 100 000 X_{ij}; i, j = 1,2, ... ,n$$

- Cada día se le transfiere aunque sea 30% de las ventas a la central:
$$T_{i} = 0.3V_{i}$$
 
- Cada tienda será visitada por el camión cuando mucho una vez:
    $$\sum_{j=1}^nX_{ij} \le 1; i =2,3, ... ,n$$    
    $$\sum_{i=1}^nX_{ij} \le 1; j =2,3, ... ,n$$   
- Adicionalmente:
$$x_{ij} \in \{0,1\}; i, j = 1,2, ... , n$$
$$F_{ij} \ge 0$$

- El dinero en tienda de cada día será dado por: $$P_i(Nuevo) = V_{i} - \frac{5}{1000}V_i\sum_{j=1}^nx_{ij} + \sum_{j=1}^nF_{ji} - \sum_{j=1}^n F_{ij} - T_i$$ donde $V_{i}$,  $F_{ij}$ y $T_i$ se refieren al dinero en tienda, flujo del día anterior y transferencias del día anterior respectivamente. adicionalmente para la solución del primer día $$P_{i}(Día0) = V_{i}$$ donde $V_{i}$ son las ventas del día previo al primer día marcado (Es decir las ventas del lunes puesto que el problema se comenzará a resolver el martes)

## Solución

Se utilizará la librería de ```pyomo```  para resolver el problema de optimización lineal, adicionalmente se utilizará el archivo dado por coppel  ```ventas.csv``` para obtener los valores de $V_i$ y un archivo adicional creado por el equipo llamado ```coordenadasCoppel.csv``` que ayudará en el cálculo de las distancias
Comenzaremos creando las matriz de costos que se utilizará para la función objetivo

In [21]:
coords = pd.read_csv('coordenadasCoppel.csv', index_col='TiendaCodigo')
coords.drop(1091, axis='index', inplace=True)
distance = {}
for i in coords.index:
    for j in coords.index:
        distance[(i,j)] = (abs(coords.loc[i,'Latitud'] - coords.loc[j,'Latitud']) + abs(coords.loc[i,'Longitud'] - coords.loc[j,'Longitud'])) * 111

Con la matriz de costos ya realizada podemos proceder a la definición de la función objetivo y variables de decisión dentro de la librería, para esto se definirá el problema comenzando por el primer registro de ventas encontrado en Lunes como día inicial (Día 0), de ahí procederá a resolverse por los siguientes 6 días de la semana comenzandoo con el martes, con los datos encontrados en ```ventas.csv```, aunado a esto se hará una pequeña correción a los datos de ventas de modo que se ignorará el departamento y se agruparán por día y por tienda

In [22]:
ventas = pd.read_csv('ventas.csv')
ventas = ventas[ventas['tipocompra'] == 'Contado']
ventas['dia_semana'] = ventas['dia_semana'].str.strip()
ventas = ventas.groupby(['dia_semana', 'num_tie'])['promedio_venta'].sum().reset_index()
ventas.set_index('num_tie', inplace = True)
ventas = ventas.sort_values(by=['dia_semana', 'num_tie'])

nodes = coords.index.tolist()

n = len(nodes)
central = nodes[0]
stores = nodes[1:]
arcs = [(i,j) for i in nodes for j in nodes if i != j]

p = dict(zip(stores, [ventas[ventas['dia_semana']=='Monday'].loc[i,'promedio_venta'] for i in stores])) 
v = dict(zip(stores, [ventas[ventas['dia_semana']=='Tuesday'].loc[i,'promedio_venta'] for i in stores]))

central_demand = np.sum([ num - 250000 for num in p if num > 250000 ])

capacity = 100000
demand = 8800
max_money = 250000

In [23]:
# ===== PYOMO MODEL =====
model = pyo.ConcreteModel()

model.x = pyo.Var(arcs, within=pyo.Binary)           # 1 if arc is used
model.f = pyo.Var(arcs, within=pyo.NonNegativeReals) # flow (amount of load carried)
model.t = pyo.Var(stores, within=pyo.NonNegativeReals)  # amount of money transferred by each store to the depot
model.r = pyo.Var(arcs, within=pyo.NonNegativeReals)  # dummy reachability flow

model.v = pyo.Param(stores, initialize= v, within=pyo.NonNegativeReals, mutable = True) #sales of each store
model.p = pyo.Param(stores, initialize= p, within=pyo.NonNegativeReals, mutable = True)  # money at each store
model.central_demand = pyo.Param(initialize= central_demand, within=pyo.NonNegativeReals, mutable = True)  # demand at the depot

# Objective function: minimize total cost
model.obj = pyo.Objective(
    expr = 25 * sum(distance[i, j] * model.x[i, j] for (i, j) in arcs) +
           (3 / 1000) * sum(model.f[i, j] for (i, j) in arcs) +
           (5 / 1000) * sum(model.v[i] * sum(model.x[i, j] for j in nodes if i!=j) for i in stores) +
           0.01 * sum(model.t[i] for i in stores),
    sense = pyo.minimize
)

# Each client must be entered at most once
model.visit_once = pyo.ConstraintList()
for i in stores:
    model.visit_once.add(expr=sum(model.x[j, i] for j in nodes if j != i) <= 1)
    model.visit_once.add(expr=sum(model.x[i, j] for j in nodes if j != i) <= 1)

# Flow constraints for demand satisfaction
model.flow_balance = pyo.ConstraintList()
for i in stores:
    inflow = sum(model.f[j, i] for j in nodes if j != i)
    outflow = sum(model.f[i, j] for j in nodes if j != i)
    model.flow_balance.add(expr= (inflow - outflow + model.p[i] >= demand))

# Flow maximum constraints (ensuring flow does not exceed maximum money)
model.flow_maximum = pyo.ConstraintList()
for dex,i in enumerate(stores):
    inflow = sum(model.f[j, i] for j in nodes if j != i)
    outflow = sum(model.f[i, j] for j in nodes if j != i)
    model.flow_maximum.add(expr= (inflow - outflow + model.p[i] - model.t[i] <= max_money))


# Flow upper bounds (no flow if no arc, and limited by capacity)
model.flow_limits = pyo.ConstraintList()
for (i, j) in arcs:
    model.flow_limits.add(expr=model.f[i, j] <= capacity * model.x[i, j])

# Depot demand constraint
total_inflow = sum(model.f[j, central] for j in stores)
model.depot_demand = pyo.Constraint(expr = total_inflow + sum(model.t[i] - 0.3*model.v[i] for i in stores)  >= central_demand)  

# Depot can't send money
model.depot_outflow = pyo.Constraint(expr = sum(model.f[central,j] for j in stores ) == 0)

# Depot transfer constraints
model.depot_transfer = pyo.ConstraintList()
for i in stores:
    model.depot_transfer.add(expr = model.t[i] >= 0.3* model.v[i])

# Solve
solver = SolverFactory('highs')  # or 'cbc', 'gurobi', etc.
result = solver.solve(
    model,
    tee=True,
    options={
        'time_limit': 600,       # Stop after 600 seconds (10 minutes)
        'mip_rel_gap': 0.05,      # Relative MIP gap ≤ 10%
        'solution_limit': 1000,  # Stop after finding 1000 valid solutions
        'mip_max_nodes': 100000  # Stop after exploring 100,000 nodes
    }
    )

Running HiGHS 1.10.0 (git hash: fd86653): Copyright (c) 2025 HiGHS under MIT licence terms


In [24]:
print('Flujos:')
for i in nodes:
        for j in nodes:
            if i != j:
                if pyo.value(model.f[i,j]) > 0:
                    print(f'Flujo de {i} a {j}: {pyo.value(model.f[i,j]): .2f}\n') 

print('\nRutas:')
for i in nodes:
    for j in nodes:
        if i != j:
            if pyo.value(model.x[i,j]) > 0.5:
                print(f'Ruta de {i} a {j}')

print('\nTransferencias:')
for i in stores:
    if pyo.value(model.t[i]) > 0:
        print(f'Tienda {i} transfiere: {pyo.value(model.t[i]): .2f} a la central')

costo_semanal = model.obj()

Flujos:
Flujo de 7824 a 512:  3649.90


Rutas:
Ruta de 7824 a 512

Transferencias:
Tienda 503 transfiere:  5656.75 a la central
Tienda 511 transfiere:  4743.78 a la central
Tienda 512 transfiere:  1585.97 a la central
Tienda 82 transfiere:  41515.20 a la central
Tienda 84 transfiere:  26512.34 a la central
Tienda 86 transfiere:  30841.39 a la central
Tienda 1050 transfiere:  13431.67 a la central
Tienda 6548 transfiere:  7175.71 a la central
Tienda 299 transfiere:  17069.64 a la central
Tienda 758 transfiere:  2489.56 a la central
Tienda 144 transfiere:  20869.93 a la central
Tienda 1159 transfiere:  14762.34 a la central
Tienda 419 transfiere:  15746.04 a la central
Tienda 1389 transfiere:  13133.94 a la central
Tienda 712 transfiere:  6417.16 a la central
Tienda 822 transfiere:  9617.27 a la central
Tienda 846 transfiere:  25911.47 a la central
Tienda 462 transfiere:  8556.11 a la central
Tienda 951 transfiere:  38004.21 a la central
Tienda 954 transfiere:  26762.90 a la central
Tien

In [25]:
days = ['Tuesday','Wednesday','Thursday','Friday','Saturday']
for day in range(1,len(days)):
    
    for i in stores:
        model.p[i] = model.p[i] + model.v[i]-0.05*(model.v[i])*sum(model.x[i,j] for j in nodes if j!=i) - model.t[i] + sum(model.f[j,i] for j in stores if j!=i) - sum(model.f[i,j] for j in stores if j!=i)
        model.v[i] = ventas[ventas['dia_semana']==days[day]].loc[i,'promedio_venta']
        
    central_demand = np.sum([ num - 250000 for num in p if num > 250000 ])
    model.central_demand = central_demand
    
    result = solver.solve(
        model,
        tee=True,
        options={
            'time_limit': 600,       # Stop after 600 seconds (10 minutes)
            'mip_rel_gap': 0.05,      # Relative MIP gap ≤ 10%
            'solution_limit': 1000,  # Stop after finding 1000 valid solutions
            'mip_max_nodes': 100000  # Stop after exploring 100,000 nodes
        }
        )
    
    print(f'\nResultados del día: {days[day]}')
    print('Flujos:')
    for i in nodes:
            for j in nodes:
                if i != j:
                    if pyo.value(model.f[i,j]) > 0:
                        print(f'Flujo de {i} a {j}: {pyo.value(model.f[i,j]): .2f}') 

    print('\nRutas:')
    for i in nodes:
        for j in nodes:
            if i != j:
                if pyo.value(model.x[i,j]) > 0.5:
                    print(f'Ruta de {i} a {j}')

    print('\nTransferencias:')
    for i in stores:
        if pyo.value(model.t[i]) > 0:
            print(f'Tienda {i} transfiere: {pyo.value(model.t[i]): .2f} a la central')
            
    costo_semanal += model.obj()


Resultados del día: Wednesday
Flujos:

Rutas:

Transferencias:
Tienda 503 transfiere:  4297.78 a la central
Tienda 511 transfiere:  3839.09 a la central
Tienda 512 transfiere:  1835.06 a la central
Tienda 82 transfiere:  39685.09 a la central
Tienda 84 transfiere:  27590.77 a la central
Tienda 86 transfiere:  27990.52 a la central
Tienda 1050 transfiere:  13575.89 a la central
Tienda 6548 transfiere:  5953.35 a la central
Tienda 299 transfiere:  19549.76 a la central
Tienda 758 transfiere:  3186.55 a la central
Tienda 144 transfiere:  25193.14 a la central
Tienda 1159 transfiere:  11791.93 a la central
Tienda 419 transfiere:  17113.60 a la central
Tienda 1389 transfiere:  14189.85 a la central
Tienda 712 transfiere:  6603.20 a la central
Tienda 822 transfiere:  9920.50 a la central
Tienda 846 transfiere:  19902.53 a la central
Tienda 462 transfiere:  8080.14 a la central
Tienda 951 transfiere:  37216.69 a la central
Tienda 954 transfiere:  23795.43 a la central
Tienda 7824 transfiere:

In [26]:
costo_semanal

26292.82379999998